# 4. Feature Engineering

This notebook creates new variables that could help the model identify risky borrowers.

The new features are based on common credit risk indicators:

- **LTV (Loan to Value):** loan amount divided by the value of the property.
- **MTV (Mortgage to Value):** amount due on the existing mortgage divided by the value of the property.
- **DTV (Debt to Value):** total debt (new loan + existing mortgage) divided by the value of the property. The higher this ratio, the less equity the client has in the house, and the lower the protection for the bank if the client defaults.
- **NEG_RECORDS:** total number of negative records in the credit history (derogatory reports + delinquent credit lines).

In [3]:
# Import the libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# Load the dataset
df = pd.read_csv("../data/HMEQ_features.csv")
df.shape

(5960, 26)

In [5]:
df.head()

,BAD,LOAN,MORTDUE,VALUE,REASON,JOB,YOJ,DEROG,DELINQ,CLAGE,...,MORTDUE_missing,YOJ_missing,NINQ_missing,CLAGE_missing,CLNO_missing,VALUE_missing,LTV,MTV,DTV,NEG_RECORDS
0,1,1100,25860.0,39025.0,HomeImp,Other,10.5,0.0,0.0,94.366667,...,0,0,0,0,0,0,0.028187,0.662652,0.690839,0.0
1,1,1300,70053.0,68400.0,HomeImp,Other,7.0,0.0,2.0,121.833333,...,0,0,0,0,0,0,0.019006,1.024167,1.043173,2.0
2,1,1500,13500.0,16700.0,HomeImp,Other,4.0,0.0,0.0,149.466667,...,0,0,0,0,0,0,0.089820,0.808383,0.898204,0.0
3,1,1500,NaN,NaN,Unknown,Unknown,NaN,NaN,NaN,NaN,...,1,1,1,1,1,1,NaN,NaN,NaN,NaN
4,0,1700,97800.0,112000.0,HomeImp,Office,3.0,0.0,0.0,93.333333,...,0,0,0,0,0,0,0.015179,0.873214,0.888393,0.0


In [6]:
# Create the new variables: Loan to Value Ratio (LTV), Mortgages to Value Ratio (MTV), and Debt to Value Ratio (DTV)
df['LTV'] = df['LOAN'] / df['VALUE']
df['MTV'] = df['MORTDUE'] / df['VALUE']
df['DTV'] = (df['LOAN'] + df['MORTDUE']) / df['VALUE']

In [7]:
df.head()

,BAD,LOAN,MORTDUE,VALUE,REASON,JOB,YOJ,DEROG,DELINQ,CLAGE,...,MORTDUE_missing,YOJ_missing,NINQ_missing,CLAGE_missing,CLNO_missing,VALUE_missing,LTV,MTV,DTV,NEG_RECORDS
0,1,1100,25860.0,39025.0,HomeImp,Other,10.5,0.0,0.0,94.366667,...,0,0,0,0,0,0,0.028187,0.662652,0.690839,0.0
1,1,1300,70053.0,68400.0,HomeImp,Other,7.0,0.0,2.0,121.833333,...,0,0,0,0,0,0,0.019006,1.024167,1.043173,2.0
2,1,1500,13500.0,16700.0,HomeImp,Other,4.0,0.0,0.0,149.466667,...,0,0,0,0,0,0,0.089820,0.808383,0.898204,0.0
3,1,1500,NaN,NaN,Unknown,Unknown,NaN,NaN,NaN,NaN,...,1,1,1,1,1,1,NaN,NaN,NaN,NaN
4,0,1700,97800.0,112000.0,HomeImp,Office,3.0,0.0,0.0,93.333333,...,0,0,0,0,0,0,0.015179,0.873214,0.888393,0.0


In [8]:
# Display the summary statistics for the new variables
df[['LTV', 'MTV', 'DTV']].describe()

,LTV,MTV,DTV
count,5848.000000,5357.000000,5357.000000
mean,0.216597,0.708494,0.914053
std,0.165112,0.381115,0.435792
min,0.015179,0.020538,0.094208
25%,0.122324,0.623000,0.838924
50%,0.170527,0.718861,0.904031
75%,0.245009,0.795614,0.963071
max,2.325000,7.162500,9.487500


In [9]:
df[['LTV', 'MTV', 'DTV']].isna().sum()

LTV    112
MTV    603
DTV    603
dtype: int64

In [10]:
df[['LOAN', 'VALUE', 'MORTDUE', 'LTV', 'MTV', 'DTV']].sort_values('LTV', ascending=False).head(20)

,LOAN,VALUE,MORTDUE,LTV,MTV,DTV
3538,18600,8000.0,57300.0,2.325000,7.162500,9.487500
4225,22000,12500.0,82000.0,1.760000,6.560000,8.320000
3757,20000,12737.0,59936.0,1.570228,4.705661,6.275889
3683,19400,12414.0,63508.0,1.562752,5.115837,6.678589
4155,21700,14580.0,60792.0,1.488340,4.169547,5.657888
3707,19500,13930.0,58638.0,1.399856,4.209476,5.609332
3655,19200,13933.0,65206.0,1.378023,4.679968,6.057992
3739,19800,14609.0,63701.0,1.355329,4.360394,5.715723
5777,46300,41600.0,39800.0,1.112981,0.956731,2.069712
5948,86000,85000.0,47355.0,1.011765,0.557118,1.568882


In [11]:
df['LTV_bin'] = pd.cut(
    df['LTV'],
    bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 1, np.inf],
    labels=['<10%', '10-20%', '20-30%', '30-40%', '40-50%', '50-100%', '>100%']
)

df.groupby('LTV_bin', observed=True)['BAD'].agg(['mean', 'count'])

,mean,count
LTV_bin,,
<10%,0.241020,863
10-20%,0.176192,2747
20-30%,0.192126,1270
30-40%,0.154034,409
40-50%,0.166667,192
50-100%,0.143662,355
>100%,0.166667,12


The relationship between `LTV` and default is weak and not monotonic. The highest default rate appears in the lowest LTV group (<10%), which mostly corresponds to small loans. This suggests that LTV alone is not a strong signal, but it may still be useful in combination with other variables.

Some observations have ratios above 100% (the debt is higher than the value of the property). These cases are possible in practice, so they are kept.

In [12]:
# Check the default rate by Debt to Value ratio (DTV)
df['DTV_bin'] = pd.cut(
    df['DTV'],
    bins=[0, 0.8, 0.9, 1, np.inf],
    labels=['<80%', '80-90%', '90-100%', '>100%']
)

df.groupby('DTV_bin', observed=True)['BAD'].agg(['mean', 'count'])

,mean,count
DTV_bin,,
<80%,0.211907,991
80-90%,0.209434,1590
90-100%,0.178228,1975
>100%,0.128589,801


The result for `DTV` is counterintuitive: clients with more debt than the value of their property (>100%) have a **lower** default rate (12.9%) than clients with a DTV below 80% (21.2%). The data does not explain why, and this pattern may not hold in other portfolios. The variable is kept and the model will decide if it is useful, but this is the type of result that should be discussed with the business before trusting it.

Number of negative records (NEG_RECORDS)

In [13]:
# Create the total number of negative records in the credit history
df['NEG_RECORDS'] = df['DEROG'] + df['DELINQ']

df.groupby('NEG_RECORDS')['BAD'].agg(['mean', 'count']).head(8)

,mean,count
NEG_RECORDS,,
0.0,0.118902,3751
1.0,0.300429,699
2.0,0.434483,290
3.0,0.513966,179
4.0,0.663158,95
5.0,0.840000,50
6.0,1.000000,41
7.0,1.000000,25


`NEG_RECORDS` combines the two strongest signals found in the EDA. Clients without any negative record have a much lower default rate than clients with one or more, and the default rate keeps increasing with each additional record.

In [14]:
# Remove the auxiliary columns used only for the analysis
df = df.drop(columns=['LTV_bin', 'DTV_bin'])

df.shape

(5960, 26)

In [15]:
# Save the dataset with the new features
df.to_csv('../data/HMEQ_features.csv', index=False)

## Feature Engineering Conclusions

- Four new variables were created: `LTV`, `MTV`, `DTV` and `NEG_RECORDS`.
- `NEG_RECORDS` shows a clear and increasing relationship with default, combining the information of `DEROG` and `DELINQ`.
- The property ratios (`LTV`, `MTV`, `DTV`) show weak or even counterintuitive relationships with default when analyzed individually (higher `DTV` is associated with a lower default rate). They are kept because tree-based models can find interactions with other variables.
- Missing values in the new variables come from missing values in `VALUE` or `MORTDUE`. They will be imputed inside the modeling pipeline.

The final dataset is saved as `HMEQ_features.csv` and will be used in the modeling stage.